# Demo 11 — Equalizzazione in cascata a 4 bande su parlato (speech.wav) e musica (acoustic.wav)

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
# 1. Catena a 4 bande per il parlato (speech.wav)
voce_sp, sr_sp = al.load_audio(AUDIO / "speech.wav", sr=SR, seconds=5.0)
stadi_speech = [
    ("Banda 1: LP Shelf -8 dB (200 Hz)", signal.tf2sos(*al.eq_shelf(200, -8.0, "low", sr=sr_sp))),
    ("Banda 2: Cut -6 dB (420 Hz)", signal.tf2sos(*al.eq_bell(420, 180, -6.0, sr=sr_sp, band_level_db=-3.0))),
    ("Banda 3: Boost +7 dB (2.8 kHz)", signal.tf2sos(*al.eq_bell(2800, 1200, 7.0, sr=sr_sp, band_level_db=3.5))),
    ("Banda 4: HP Shelf +4 dB (6 kHz)", signal.tf2sos(*al.eq_shelf(6000, 4.0, "high", sr=sr_sp))),
]
sos_speech_tot = np.vstack([sos for _, sos in stadi_speech])
voce_sp_eq = al.filter_sos(sos_speech_tot, voce_sp)
voce_sp_eq_rms = voce_sp_eq * (al.rms(voce_sp) / max(al.rms(voce_sp_eq), 1e-12))

# 2. Catena a 4 bande per la musica (acoustic.wav)
musica_ac, sr_ac = al.load_audio(AUDIO / "acoustic.wav", sr=SR, seconds=6.0)
stadi_musica = [
    ("Banda 1: LP Shelf +6.5 dB (140 Hz)", signal.tf2sos(*al.eq_shelf(140, 6.5, "low", sr=sr_ac))),
    ("Banda 2: Cut -7.5 dB (850 Hz)", signal.tf2sos(*al.eq_bell(850, 450, -7.5, sr=sr_ac, band_level_db=-3.8))),
    ("Banda 3: Boost +5.5 dB (3.5 kHz)", signal.tf2sos(*al.eq_bell(3500, 1500, 5.5, sr=sr_ac, band_level_db=2.8))),
    ("Banda 4: HP Shelf +6 dB (8 kHz)", signal.tf2sos(*al.eq_shelf(8000, 6.0, "high", sr=sr_ac))),
]
sos_musica_tot = np.vstack([sos for _, sos in stadi_musica])
musica_ac_eq = al.filter_sos(sos_musica_tot, musica_ac)
musica_ac_eq_rms = musica_ac_eq * (al.rms(musica_ac) / max(al.rms(musica_ac_eq), 1e-12))

f_casc = np.geomspace(30, 18000, 1500)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Curva EQ Speech (speech.wav)", "Curva EQ Musica (acoustic.wav)"])
for col, (stadi, sos_tot, sr_cur) in enumerate(((stadi_speech, sos_speech_tot, sr_sp),
                                                (stadi_musica, sos_musica_tot, sr_ac)), 1):
    f_lim = f_casc[f_casc < 0.49 * sr_cur]
    for nome, sos in stadi:
        _, H = signal.sosfreqz(sos, worN=f_lim, fs=sr_cur)
        fig.add_trace(go.Scatter(x=f_lim, y=al.to_db(np.abs(H), floor=1e-6), name=nome,
                                 line=dict(dash="dot"), showlegend=(col == 1)), row=1, col=col)
    _, H_tot = signal.sosfreqz(sos_tot, worN=f_lim, fs=sr_cur)
    fig.add_trace(go.Scatter(x=f_lim, y=al.to_db(np.abs(H_tot), floor=1e-6), name=f"Totale (col {col})",
                             line=dict(width=3)), row=1, col=col)
fig.update_xaxes(type="log", title_text="Frequenza [Hz]")
fig.update_yaxes(title_text="Guadagno [dB]", range=[-12, 10])
fig.update_layout(height=390)
fig.show()

print("Confronto 1 — Parlato (speech.wav):")
al.compare_audio([("Speech originale", voce_sp),
                  ("Speech equalizzato (4 bande)", voce_sp_eq_rms)], sr_sp)
print("Confronto 2 — Musica (acoustic.wav):")
al.compare_audio([("Musica originale", musica_ac),
                  ("Musica equalizzata (Warmth & Air)", musica_ac_eq_rms)], sr_ac)
